# Passing Quality — Final Model

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VISUAL DESIGN SYSTEM
# Style: The Athletic / FiveThirtyEight editorial
# ═══════════════════════════════════════════════════════════════════

from matplotlib.offsetbox import AnchoredText
import matplotlib.patheffects as pe

# ── 1. Color palette ──
# Background & structure
BG       = '#FAFAFA'
GRID     = '#EDEDED'
AXIS     = '#D5D5D5'
TEXT     = '#2D2D2D'
SUBTEXT  = '#737373'

# Semantic colors (consistent across ALL charts)
C_PRE      = '#5D6D7E'   # Pre-transfer: cool slate
C_POST     = '#1A9C6E'   # Actual post-transfer: confident green
C_BASELINE = '#BF5B3F'   # Baseline model: warm terracotta
C_TACTICAL = '#2E74B5'   # Tactical model: strong blue
C_POP      = '#C8DCC0'   # Population distribution: muted sage

# ── 2. Typography & layout ──
plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   BG,
    'axes.edgecolor':   AXIS,
    'axes.labelcolor':  SUBTEXT,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.titlepad':    16,
    'axes.labelsize':   9.5,
    'axes.grid':        False,
    'text.color':       TEXT,
    'xtick.color':      SUBTEXT,
    'ytick.color':      SUBTEXT,
    'xtick.labelsize':  8.5,
    'ytick.labelsize':  8.5,
    'xtick.major.size': 0,
    'ytick.major.size': 0,
    'xtick.major.pad':  6,
    'ytick.major.pad':  6,
    'font.family':      'sans-serif',
    'figure.dpi':       140,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
})

def add_subtitle(ax, text, y=1.02):
    ax.text(0, y, text, transform=ax.transAxes,
            fontsize=9, color=SUBTEXT, style='italic')

def stat_box(ax, lines, loc='upper left'):
    txt = '\n'.join(lines)
    anchored = AnchoredText(txt, loc=loc, frameon=True,
        prop=dict(fontsize=9, fontfamily='monospace', color=TEXT))
    anchored.patch.set_boxstyle('round,pad=0.4')
    anchored.patch.set_facecolor('white')
    anchored.patch.set_edgecolor(GRID)
    anchored.patch.set_alpha(0.9)
    ax.add_artist(anchored)

In [ ]:
DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
RAW_DIR  = "../../../../thesis_data/raw_data/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

# Target
mf["delta_PQ"] = mf["to_Passing quality"] - mf["from_Passing quality"]

# Delta team qualities
for tq in ["ATTACK", "DEFENCE", "OUTCOME"]:
    mf[f"delta_tq_{tq}"] = mf[f"to_q_{tq}"] - mf[f"from_q_proj_{tq}"]

DELTA_TQ = ["delta_tq_ATTACK", "delta_tq_DEFENCE", "delta_tq_OUTCOME"]
mf = mf.dropna(subset=["from_Passing quality", "delta_PQ"] + DELTA_TQ)

# Team name lookup (for case study)
teams = pd.read_parquet(RAW_DIR + "Wyscout/wyscout_teams.parquet")
team_map = dict(zip(teams['team_id'], teams['name']))

# Split
train, test = train_test_split(mf, test_size=0.2, random_state=42)
y_train = train["delta_PQ"]
y_test  = test["delta_PQ"]
print(f"Midfielders: {len(mf):,}  (train: {len(train):,}, test: {len(test):,})")

---
## Baseline: Pre-Transfer Quality Only

The simplest hypothesis: knowing a player's passing quality before the transfer
is enough to predict how much it will change. This captures regression to the mean
(extreme values tend to move toward the center) but ignores tactical context entirely.

In [ ]:
X_tr = sm.add_constant(train[["from_Passing quality"]])
X_te = sm.add_constant(test[["from_Passing quality"]])

baseline = sm.OLS(y_train, X_tr).fit()
baseline_pred = baseline.predict(X_te)

print(baseline.summary())

---
## Tactical Model: Pre-Quality + Team Style Changes

Does knowing how the tactical environment changed improve the prediction?
We add three delta team qualities selected via exhaustive search (127 combinations):

- **ATTACK**: direct vs possession-based build-up
- **DEFENCE**: high press vs low block
- **OUTCOME**: team performance (xPts, points)

In [ ]:
feat = ["from_Passing quality"] + DELTA_TQ
X_tr = sm.add_constant(train[feat])
X_te = sm.add_constant(test[feat])

tactical = sm.OLS(y_train, X_tr).fit()
tactical_pred = tactical.predict(X_te)

print(tactical.summary())

---
## Results

In [ ]:
b_r2  = r2_score(y_test, baseline_pred)
b_mae = mean_absolute_error(y_test, baseline_pred)
t_r2  = r2_score(y_test, tactical_pred)
t_mae = mean_absolute_error(y_test, tactical_pred)

comp = pd.DataFrame({
    "Model": ["Baseline (from_PQ only)", "Tactical (from_PQ + 3 delta TQ)"],
    "R2_test": [b_r2, t_r2],
    "MAE_test": [b_mae, t_mae],
    "R2_adj_train": [baseline.rsquared_adj, tactical.rsquared_adj],
    "BIC": [baseline.bic, tactical.bic],
})
print(comp.to_string(index=False))

print(f"\nR2 gain from tactical context: {t_r2 - b_r2:+.4f}")
print(f"MAE reduction:                 {b_mae - t_mae:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
fig.subplots_adjust(wspace=0.32)

configs = [
    ('Baseline Model', baseline_pred, b_r2, b_mae, C_BASELINE),
    ('Tactical Model', tactical_pred, t_r2, t_mae, C_TACTICAL),
]

for ax, (name, pred, r2, mae, color) in zip(axes, configs):
    # Grid first
    ax.grid(True, color=GRID, linewidth=0.6, alpha=0.8)

    # Ideal line
    lims = [min(y_test.min(), pred.min()) - 0.1, max(y_test.max(), pred.max()) + 0.1]
    ax.plot(lims, lims, color=SUBTEXT, linewidth=0.8, linestyle='--', alpha=0.4, zorder=2)

    # Points
    ax.scatter(y_test, pred, alpha=0.10, s=10, color=color, edgecolors='none', zorder=3)

    ax.set_xlabel('Actual delta')
    ax.set_ylabel('Predicted delta')
    ax.set_title(name)
    ax.set_aspect('equal')
    ax.spines['left'].set_color(AXIS)
    ax.spines['bottom'].set_color(AXIS)

    # Stat box
    stat_box(ax, ['R²  = {:.3f}'.format(r2), 'MAE = {:.3f}'.format(mae)])

plt.tight_layout()
plt.show()

---
## Coefficient Interpretation

In [ ]:
coefs = pd.DataFrame({
    "Feature": tactical.params.index,
    "Coefficient": tactical.params.values,
    "p-value": tactical.pvalues.values,
})
coefs = coefs[coefs["Feature"] != "const"]

print(coefs.to_string(index=False))

In [ ]:
# ── Rice data ──
rice = mf[mf['wy_player_id'] == 379209].iloc[0]
from_team = team_map.get(rice['from_team_id'], 'Unknown')
to_team   = team_map.get(rice['to_team_id'], 'Unknown')
pre_pq    = rice['from_Passing quality']
post_pq   = rice['to_Passing quality']
actual_delta = post_pq - pre_pq

# Style descriptions
style_labels = {
    'ATTACK':  ('Direct long balls',    'Patient build-up'),
    'DEFENCE': ('Low block',            'High press'),
    'OUTCOME': ('Lower table',          'Top of table'),
}

qualities = ['ATTACK', 'DEFENCE', 'OUTCOME']
fig, ax = plt.subplots(figsize=(10, 3.5))

for i, tq in enumerate(qualities):
    fr = rice['from_q_proj_' + tq]
    to = rice['to_q_' + tq]
    ax.plot([fr, to], [i, i], color=GRID, linewidth=4, zorder=1, solid_capstyle='round')
    ax.scatter(fr, i, color=C_PRE, s=130, zorder=3, edgecolors='white', linewidth=1.8)
    ax.scatter(to, i, color=C_POST, s=130, zorder=3, edgecolors='white', linewidth=1.8)

# Style labels at edges
for i, tq in enumerate(qualities):
    left_label, right_label = style_labels[tq]
    ax.text(-2.15, i + 0.28, left_label, ha='left', va='bottom',
            fontsize=7.5, color=SUBTEXT, style='italic')
    ax.text(2.15, i + 0.28, right_label, ha='right', va='bottom',
            fontsize=7.5, color=SUBTEXT, style='italic')

# Team legend top-right
ax.scatter([], [], color=C_PRE, s=60, edgecolors='white')
ax.scatter([], [], color=C_POST, s=60, edgecolors='white')
ax.text(1.0, 1.08, from_team, transform=ax.transAxes,
        ha='right', va='top', fontsize=9.5, fontweight='bold', color=C_PRE)
ax.text(1.0, 0.96, to_team, transform=ax.transAxes,
        ha='right', va='top', fontsize=9.5, fontweight='bold', color=C_POST)

ax.set_yticks(list(range(len(qualities))))
ax.set_yticklabels(qualities, fontsize=10.5, fontweight='bold')
ax.axvline(0, color=SUBTEXT, linewidth=0.6, linestyle=':', alpha=0.4)
ax.set_xlim(-2.3, 2.3)
ax.set_title('How the tactical environment changed')
ax.invert_yaxis()
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', length=0, labelbottom=False)
ax.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Predictions ──
rice_base_x = pd.DataFrame({'const': [1.0], 'from_Passing quality': [pre_pq]})

rice_tact_x = pd.DataFrame({
    'const': [1.0],
    'from_Passing quality': [pre_pq],
    'delta_tq_ATTACK':  [rice['delta_tq_ATTACK']],
    'delta_tq_DEFENCE': [rice['delta_tq_DEFENCE']],
    'delta_tq_OUTCOME': [rice['delta_tq_OUTCOME']],
})

pred_base = baseline.predict(rice_base_x)[0]
pred_tact = tactical.predict(rice_tact_x)[0]
to_base = pre_pq + pred_base
to_tact = pre_pq + pred_tact

# ── Strip chart ──
all_pq = mf['from_Passing quality'].tolist() + mf['to_Passing quality'].tolist()

fig, ax = plt.subplots(figsize=(12, 3.3))

# Background strip
ax.axhline(0, color=GRID, linewidth=30, solid_capstyle='round', zorder=0, alpha=0.6)

# Population dots
np.random.seed(0)
jitter = np.random.normal(0, 0.008, len(all_pq))
ax.scatter(all_pq, jitter, alpha=0.04, s=30, color=C_POP, edgecolors='none', zorder=1)

# Arrow from pre to post
ax.annotate('', xy=(post_pq, 0), xytext=(pre_pq, 0),
            arrowprops=dict(arrowstyle='->', color=TEXT, lw=2, mutation_scale=14),
            zorder=4)

# All markers on the same y=0 line
# Actual: circles (darker)
ax.scatter(pre_pq, 0, s=240, color=C_PRE, zorder=6, marker='o',
           edgecolors='white', linewidth=2.2)
ax.scatter(post_pq, 0, s=240, color=C_POST, zorder=6, marker='o',
           edgecolors='white', linewidth=2.2)

# Predictions: diamonds (lighter tones of same palette)
ax.scatter(to_base, 0, s=160, color='#D4A59A', zorder=5, marker='D',
           edgecolors='white', linewidth=1.8)
ax.scatter(to_tact, 0, s=160, color='#7FB3D3', zorder=5, marker='D',
           edgecolors='white', linewidth=1.8)

# Delta label on the arrow
mid_x = (pre_pq + post_pq) / 2
ax.text(mid_x, 0.04, '{:+.3f}'.format(post_pq - pre_pq),
        ha='center', va='bottom', fontsize=9, fontweight='bold', color=TEXT)

# ── Compact legend ──
leg_x = 0.98
leg_items = [
    ('o', C_PRE,     'Pre  {:.3f}'.format(pre_pq)),
    ('o', C_POST,    'Post  {:.3f}'.format(post_pq)),
    ('D', '#D4A59A', 'Baseline pred  {:.3f}'.format(to_base)),
    ('D', '#7FB3D3', 'Tactical pred  {:.3f}'.format(to_tact)),
]

for i, (marker, color, label) in enumerate(leg_items):
    y = 0.92 - i * 0.18
    ax.scatter(leg_x, y, s=50, color=color, marker=marker,
              edgecolors='white', linewidth=1, transform=ax.transAxes,
              clip_on=False, zorder=10)
    ax.text(leg_x - 0.015, y, label,
            ha='right', va='center', fontsize=8, color=SUBTEXT,
            fontweight='bold', transform=ax.transAxes, clip_on=False)

ax.set_xlim(-1, 1)
ax.set_ylim(-0.08, 0.1)
ax.set_title('Where does Rice land after the transfer?')
add_subtitle(ax, 'Passing Quality across all midfielders')
ax.set_xlabel('Passing Quality (z-score)')
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_color(AXIS)
ax.grid(False)
plt.tight_layout()
plt.show()